In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
customers = pd.read_csv(RAW_PATH / 'olist_customers_dataset.csv')
geolocation = pd.read_csv(RAW_PATH / 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(RAW_PATH / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_PATH / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW_PATH / 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(RAW_PATH / 'olist_orders_dataset.csv')
products = pd.read_csv(RAW_PATH / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW_PATH / 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(RAW_PATH / 'product_category_name_translation.csv')

In [4]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

In [5]:
def clean_text(series, case='title'):
    series = series.astype('string').str.strip()
    
    if case == 'upper':
        return series.str.upper()
    
    if case == 'lower':
        return series.str.lower()
    
    if case == 'title':
        return series.str.title()
    
    return series

In [6]:
customers_clean['customer_city'] = clean_text(customers_clean['customer_city'], case='title')
customers_clean['customer_state'] = clean_text(customers_clean['customer_state'], case='upper')

In [7]:
order_date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in order_date_columns:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

In [8]:
orders_clean['order_purchase_date'] = orders_clean['order_purchase_timestamp'].dt.normalize()
orders_clean['order_year'] = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['order_month'] = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['order_year_month'] = orders_clean['order_purchase_timestamp'].dt.to_period('M').astype(str)

In [9]:
orders_clean['is_delivered'] = np.where(
    orders_clean['order_status'] == 'delivered',
    1,
    0
)

orders_clean['is_canceled'] = np.where(
    orders_clean['order_status'] == 'canceled',
    1,
    0
)

In [10]:
orders_clean['approval_time_hours'] = (
    (orders_clean['order_approved_at'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 3600
).round(2)

orders_clean['delivery_time_days'] = (
    (orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 86400
).round(2)

orders_clean['estimated_delivery_time_days'] = (
    (orders_clean['order_estimated_delivery_date'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 86400
).round(2)

orders_clean['delivery_delay_days'] = (
    (orders_clean['order_delivered_customer_date'] - orders_clean['order_estimated_delivery_date'])
    .dt.total_seconds() / 86400
).round(2)

In [11]:
orders_clean['delivery_status'] = np.select(
    [
        orders_clean['order_delivered_customer_date'].isna(),
        orders_clean['delivery_delay_days'] > 0,
        orders_clean['delivery_delay_days'] <= 0
    ],
    [
        'not_delivered',
        'late',
        'on_time_or_early'
    ],
    default='unknown'
)

orders_clean['is_late'] = np.where(
    orders_clean['delivery_status'] == 'late',
    1,
    np.where(
        orders_clean['delivery_status'] == 'on_time_or_early',
        0,
        np.nan
    )
)

In [12]:
order_items_clean['shipping_limit_date'] = pd.to_datetime(
    order_items_clean['shipping_limit_date'],
    errors='coerce'
)

order_items_clean['item_total_value'] = (
    order_items_clean['price'] + order_items_clean['freight_value']
).round(2)

order_items_clean['item_count'] = 1

In [13]:
payments_clean['payment_installment_group'] = np.select(
    [
        payments_clean['payment_installments'] == 1,
        payments_clean['payment_installments'].between(2, 6),
        payments_clean['payment_installments'] >= 7
    ],
    [
        '1x',
        '2x_to_6x',
        '7x_or_more'
    ],
    default='unknown'
)

In [14]:
products_clean = products_clean.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

In [15]:
products_clean = products_clean.merge(
    category_translation_clean,
    on='product_category_name',
    how='left'
)

In [16]:
manual_category_translation = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_and_food_preparers'
}

products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english']
    .fillna(products_clean['product_category_name'].map(manual_category_translation))
    .fillna('unknown')
)

products_clean['product_category_name'] = (
    products_clean['product_category_name']
    .fillna('unknown')
)

In [17]:
dimension_columns = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

products_clean['product_volume_cm3'] = (
    products_clean['product_length_cm'] *
    products_clean['product_height_cm'] *
    products_clean['product_width_cm']
)

products_clean['has_missing_product_dimension'] = (
    products_clean[dimension_columns].isna().any(axis=1)
)

In [18]:
reviews_clean = reviews_clean.reset_index(drop=True)
reviews_clean.insert(0, 'review_key', reviews_clean.index + 1)

review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(reviews_clean[col], errors='coerce')

reviews_clean['has_review_comment'] = (
    reviews_clean['review_comment_message']
    .astype('string')
    .str.strip()
    .notna()
    &
    (reviews_clean['review_comment_message'].astype('string').str.strip() != '')
)

reviews_clean['review_response_time_days'] = (
    (reviews_clean['review_answer_timestamp'] - reviews_clean['review_creation_date'])
    .dt.total_seconds() / 86400
).round(2)

reviews_clean['review_comment_title'] = reviews_clean['review_comment_title'].fillna('')
reviews_clean['review_comment_message'] = reviews_clean['review_comment_message'].fillna('')

In [19]:
geolocation_clean['geolocation_city'] = clean_text(
    geolocation_clean['geolocation_city'],
    case='title'
)

geolocation_clean['geolocation_state'] = clean_text(
    geolocation_clean['geolocation_state'],
    case='upper'
)

geolocation_clean = geolocation_clean.drop_duplicates()

In [20]:
def first_mode(series):
    mode_values = series.mode(dropna=True)
    
    if len(mode_values) > 0:
        return mode_values.iloc[0]
    
    return np.nan

In [21]:
geolocation_clean = (
    geolocation_clean
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': first_mode,
        'geolocation_state': first_mode
    })
)

In [22]:
print('Linhas geolocation original:', len(geolocation))
print('Linhas geolocation limpa:', len(geolocation_clean))
print('Prefixos únicos na geolocation limpa:', geolocation_clean['geolocation_zip_code_prefix'].nunique())
print('Duplicatas na geolocation limpa:', geolocation_clean.duplicated().sum())

Linhas geolocation original: 1000163
Linhas geolocation limpa: 19015
Prefixos únicos na geolocation limpa: 19015
Duplicatas na geolocation limpa: 0


In [23]:
clean_tables = {
    'customers_clean': customers_clean,
    'sellers_clean': sellers_clean,
    'products_clean': products_clean,
    'orders_clean': orders_clean,
    'order_items_clean': order_items_clean,
    'payments_clean': payments_clean,
    'reviews_clean': reviews_clean,
    'geolocation_clean': geolocation_clean
}

In [24]:
clean_summary = []

for table_name, df in clean_tables.items():
    clean_summary.append({
        'table_name': table_name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'duplicated_rows': df.duplicated().sum(),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 / 1024, 2)
    })

clean_summary_df = pd.DataFrame(clean_summary).sort_values(by='rows', ascending=False)

clean_summary_df

,table_name,rows,columns,duplicated_rows,memory_mb
4,order_items_clean,112650,9,0,31.26
5,payments_clean,103886,6,0,21.60
0,customers_clean,99441,5,0,26.59
3,orders_clean,99441,20,0,42.86
6,reviews_clean,99224,10,0,33.93
2,products_clean,32951,12,0,8.54
7,geolocation_clean,19015,5,0,2.44
1,sellers_clean,3095,4,0,0.59


In [25]:
clean_null_summary = []

for table_name, df in clean_tables.items():
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_percentage = round((null_count / len(df)) * 100, 2)
        
        if null_count > 0:
            clean_null_summary.append({
                'table_name': table_name,
                'column': column,
                'null_count': null_count,
                'null_percentage': null_percentage
            })

clean_null_summary_df = pd.DataFrame(clean_null_summary).sort_values(
    by=['table_name', 'null_count'],
    ascending=[True, False]
)

clean_null_summary_df

,table_name,column,null_count,null_percentage
10,orders_clean,order_delivered_customer_date,2965,2.98
12,orders_clean,delivery_time_days,2965,2.98
13,orders_clean,delivery_delay_days,2965,2.98
14,orders_clean,is_late,2965,2.98
9,orders_clean,order_delivered_carrier_date,1783,1.79
8,orders_clean,order_approved_at,160,0.16
11,orders_clean,approval_time_hours,160,0.16
0,products_clean,product_name_length,610,1.85
1,products_clean,product_description_length,610,1.85
2,products_clean,product_photos_qty,610,1.85


In [26]:
orders_clean[
    [
        'order_id',
        'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'delivery_time_days',
        'delivery_delay_days',
        'delivery_status',
        'is_late'
    ]
].head()

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,delivery_status,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,on_time_or_early,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,on_time_or_early,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,on_time_or_early,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,on_time_or_early,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,on_time_or_early,0.0


In [27]:
customers_clean.to_csv(PROCESSED_PATH / 'customers_clean.csv', index=False)
sellers_clean.to_csv(PROCESSED_PATH / 'sellers_clean.csv', index=False)
products_clean.to_csv(PROCESSED_PATH / 'products_clean.csv', index=False)
orders_clean.to_csv(PROCESSED_PATH / 'orders_clean.csv', index=False)
order_items_clean.to_csv(PROCESSED_PATH / 'order_items_clean.csv', index=False)
payments_clean.to_csv(PROCESSED_PATH / 'payments_clean.csv', index=False)
reviews_clean.to_csv(PROCESSED_PATH / 'reviews_clean.csv', index=False)
geolocation_clean.to_csv(PROCESSED_PATH / 'geolocation_clean.csv', index=False)

In [28]:
list(PROCESSED_PATH.glob('*.csv'))

[WindowsPath('../data/processed/customers_clean.csv'),
 WindowsPath('../data/processed/geolocation_clean.csv'),
 WindowsPath('../data/processed/orders_clean.csv'),
 WindowsPath('../data/processed/order_items_clean.csv'),
 WindowsPath('../data/processed/payments_clean.csv'),
 WindowsPath('../data/processed/products_clean.csv'),
 WindowsPath('../data/processed/reviews_clean.csv'),
 WindowsPath('../data/processed/sellers_clean.csv')]